# 04 · Statistical tests, SQL analysis and business insights
**Steps 9–11.** Each test: hypothesis → test → assumptions → p-value → decision → business insight.

In [ ]:
import sys
sys.path.append("..")  # so `src` can be imported from the notebooks folder

import pandas as pd
pd.set_option("display.max_columns", 50)
from src.db import get_engine
from src.eda import load_order_features
from src.stats_tests import ttest_delay_vs_review, anova_category_vs_order_value, chi_square_payment_vs_status
from src.queries import QUERIES, run_query

engine = get_engine()
of = load_order_features(engine)

def show_test(r):
    print(r["name"])
    print("H0:", r["h0"])
    print("H1:", r["h1"])
    print("Test:", r["test"])
    print("Assumptions:")
    for a in r["assumptions"]:
        print("  -", a)
    display(pd.Series(r["results"], name="value").to_frame())
    print("Decision:", r["decision"])
    print("Insight:", r["insight"])

## T-test: do delayed orders get lower review scores?

In [ ]:
show_test(ttest_delay_vs_review(of))

## ANOVA: does order value differ across categories?

In [ ]:
anova = anova_category_vs_order_value(of)
show_test(anova)
anova["means_table"]

## Chi-square: is payment method associated with order status?

In [ ]:
chi = chi_square_payment_vs_status(of)
show_test(chi)
chi["contingency_pct"]

## Step 10: SQL business analysis
All queries are in `src/queries.py` (and exported to `sql/business_queries.sql`). Run any of them for the full period:

In [ ]:
lo, hi = of["purchase_month"].min(), of["purchase_month"].max()
for name, (description, _) in QUERIES.items():
    print(f"{name:26s} {description}")

In [ ]:
print(QUERIES["top_sellers"][1])
run_query(engine, "top_sellers", lo, hi)

In [ ]:
run_query(engine, "monthly_revenue", lo, hi)

In [ ]:
run_query(engine, "delay_vs_review", lo, hi)

## Step 11: Business insights
Run `python run_pipeline.py` to generate `reports/business_insights.md`, then rewrite each insight in your own words using the pattern **Observation → Interpretation → Business impact**.